<!-- gs-watermark -->
> **© 2026 Gokaran Shukla** · part of [symphase](https://github.com/Gokaran-Shukla/symphase) · MIT License — any reuse must keep this copyright notice and credit the author.

# Quiz 11 — self-consistent magnon decay (20 questions)

Work through **26_tutorial_self_consistent_decay.ipynb** first. Questions 6–12 are hands-on.

**How to answer:** replace `...` by your answer and run the cell — a letter for multiple choice, a number or a text otherwise. `q.hint(n)`, `q.reveal(n)`, `q.score()`.
Prefer clicking? use `q.widget(n)` (needs `ipywidgets`).

In [1]:
# run this first (works from the repository's notebooks/ folder)
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # use the local package if not pip-installed
EX = os.path.abspath("../examples")
%matplotlib inline

In [2]:
from symphase.quiz import Quiz
q = Quiz('scdecay')
print(q.title, '-', len(q), 'questions')

Quiz 11 — self-consistent magnon decay - 20 questions


### Helper cell for the hands-on questions

In [3]:
import math, numpy as np, spglib
import matplotlib.pyplot as plt
from symphase.magint import FourMagnonHF, MagnonSelfEnergy
from symphase.magcheck import BosonCluster, classical_expansion
from symphase.magnons import relax_spins, make_path, analyse
from symphase.tensors import read_poscar, read_incar
a = 3.0

def canted_chain(h=4.2):
    """AFM chain, XYZ exchange (Jy = 1.3), field h along −z: canted, gapped — the ED test bed"""
    J = np.diag([1.0, 1.3, 1.0]); S = np.array([1.0, 1.0])
    pairs = [(0, 1, np.array([a, 0, 0.]), J), (1, 0, np.array([a, 0, 0.]), J)]
    hfield = np.array([[0, 0, -h]] * 2)
    dirs, _, _ = relax_spins(np.array([[0, 1, 0.5], [0, -1, 0.5]]), S, [0, 1], pairs, [None, None], hfield=hfield,
                             steps=200000, tol=1e-15)
    return FourMagnonHF(np.diag([2 * a, 12, 12]), np.array([[0, 0, 0], [a, 0, 0.]]), [0, 1], dirs, S, pairs, None, hfield)

def triangular(S=0.5, G=36):
    """120° triangular AFM (√3×√3 cell of examples/vasp/Triangular_120AFM_INCAR), J = 1 meV"""
    d = "../examples/vasp/Triangular_120AFM_INCAR"
    p = read_poscar(f"{d}/POSCAR"); mags = np.array(read_incar(f"{d}/INCAR")["magmom"]).reshape(-1, 3)
    ds = spglib.get_symmetry_dataset((p["lattice"], p["frac"], [0, 0, 0]))
    m = analyse(p["lattice"], p["frac"], [0, 0, 0], ["Cr"] * 3, mags, ds.rotations, ds.translations, J=[1.0], spin=S,
                rmax=3.1, return_model=True)
    hf = FourMagnonHF.from_model(np.asarray(p["lattice"], float), np.asarray(p["frac"], float), m).solve(grid=(G, G, 1))
    return hf, MagnonSelfEnergy(hf)

def canted_square(hr, S=1.0):
    """square-lattice Heisenberg AFM (J = 1 meV) in a field h = hr·h_s along z (h_s = 8JS), √2 cell"""
    lat = np.array([[a, a, 0], [a, -a, 0], [0, 0, 12.]])
    pairs = [(0, 1, np.array(d, float), np.eye(3)) for d in ([a, 0, 0], [-a, 0, 0], [0, a, 0], [0, -a, 0])]
    s = math.sqrt(1 - hr ** 2)
    hf = FourMagnonHF(lat, np.array([[0, 0, 0], [a, 0, 0.]]), [0, 1], np.array([[s, 0, hr], [-s, 0, hr]]),
                      np.array([S, S]), pairs, None, np.array([[0, 0, -8 * hr * S]] * 2))
    return hf, MagnonSelfEnergy(hf), lat

def square_fm(J=-2.0, K=0.5, S=1.0):
    """square-lattice ferromagnet, easy axis z (gap 2KS)"""
    pairs = [(0, 0, np.array([a, 0, 0.]), J * np.eye(3)), (0, 0, np.array([0, a, 0.]), J * np.eye(3))]
    hf = FourMagnonHF(np.diag([a, a, 12.]), np.zeros((1, 3)), [0], np.array([[0, 0, 1.]]), np.array([S]), pairs,
                      [np.diag([0, 0, -K])], None)
    return hf, MagnonSelfEnergy(hf)

def ed_check(h=4.2, nmax=8):
    """exact diagonalisation (3-cell cluster) vs bubble + tadpole and four-magnon sunset; returns rows per mode"""
    hf = canted_chain(h); se = MagnonSelfEnergy(hf)
    cl = BosonCluster(hf, 3, nmax); qs = cl.kpoints(); k = qs[1]; e = se._mset(k)[0]
    H2 = cl.quadratic()
    ed3 = cl.second_order(H2, cl.operator(se.terms3), e)
    hf.update(qs, 0.0); ed4 = cl.second_order(H2, cl.operator(se.terms4) - cl.operator(cl.hf_terms()), e); hf.hf = None
    rows = []
    for n in range(2):
        _, w, ee = se.cubic_poles(k, n, 0.0, qs=qs); bub = float(np.sum(w / (e[n] - ee)).real)
        tad = se.tadpole(k, n, 0.0, ks=qs)
        _, w, ee = se.quartic_poles(k, n, 0.0, qs=qs); sun = float(np.sum(w / (e[n] - ee)).real)
        rows.append(dict(n=n, eps=e[n], ed3=ed3[n]["second"], bubble=bub, tadpole=tad, ed4=ed4[n]["second"], sunset=sun))
    return rows

def tri_velocity(se, hf, kf=0.03):
    """2S·δε/ε of the three Goldstone branches at small k (x direction): HF (first order), Re Σ³, total"""
    k = np.array([kf * 2 * np.pi / a, 0, 0])
    o = se.onshell(k, 0.0, (36, 36, 1), eta=0.01)
    dhf = hf.first_order_shift([k])[0]
    S2 = 2 * hf.S[0]
    return dict(e0=o["e0"], hf=S2 * dhf / o["e0"], re3=S2 * o["re3"] / o["e0"],
                total=S2 * (dhf + o["re3"] + o["tad"]) / o["e0"], gamma3=o["gamma3"])

from symphase.magint import SelfConsistentDecay

def sc_square(S=1.0, hr=0.85, G=10, eta=0.05):
    """canted square AFM in a field, self-consistent decay object on a closed G×G grid (η = eta·S)"""
    hfq, seq, latq = canted_square(hr, S=S)
    return seq, SelfConsistentDecay(seq, grid=(G, G, 1), eta=eta * S), latq

def born_vs_sc(sc):
    """(Born Γ, self-consistent Γ) on the grid (Goldstone points removed)"""
    sc.solve("onshell"); g0 = sc.G.copy()
    sc.solve("sc", mix=0.3, maxiter=300, tol=1e-6 * max(1.0, sc.e0.max() / 6.8))
    m = ~sc.gold
    return g0[m], sc.G[m].copy()


### Question 1
Why does the Born (on-shell) decay rate fail near decay thresholds?

* **(a)** it is too small
* **(b)** the two-magnon continuum has van Hove edges: Γ_Born is singular there and can exceed ε
* **(c)** it ignores temperature
* **(d)** it is exact

In [4]:
q.answer(1, ...)

Q1: replace the placeholder with your answer, e.g. q.answer(1, 'b')


### Question 2
What is the iΓ approximation?

* **(a)** Γ = 0
* **(b)** internal magnons carry their own damping: ω − ε_q − ε_{k−q} + i(Γ_q + Γ_{k−q})/2, iterated to self-consistency
* **(c)** a Lorentzian fit
* **(d)** a phonon bath

In [5]:
q.answer(2, ...)

Q2: replace the placeholder with your answer, e.g. q.answer(2, 'b')


### Question 3
Why must the grid be closed (unshifted)?

* **(a)** speed
* **(b)** k, q and k−q must all be grid points, so the internal widths Γ_q are known
* **(c)** to include temperature
* **(d)** symmetry

In [6]:
q.answer(3, ...)

Q3: replace the placeholder with your answer, e.g. q.answer(3, 'b')


### Question 4
Why is the energy kept at the 1/S on-shell value instead of solving the Dyson equation off-shell?

* **(a)** it is cheaper
* **(b)** off-shell Re Σ mixes orders in 1/S and breaks the Hartree–Fock/cubic Goldstone cancellation of non-collinear magnets
* **(c)** Dyson has no solution
* **(d)** for finite T

In [7]:
q.answer(4, ...)

Q4: replace the placeholder with your answer, e.g. q.answer(4, 'b')


### Question 5
Why are the source (all-annihilation) poles not broadened?

* **(a)** they are zero
* **(b)** they are anti-resonant (negative energy): broadening them would leak an unphysical Im Σ at ω > 0
* **(c)** for speed
* **(d)** they are broadened

In [8]:
q.answer(5, ...)

Q5: replace the placeholder with your answer, e.g. q.answer(5, 'b')


### Question 6
Hands-on: sc_square(1.0) — Born maximum Γ on the grid (meV, 2 decimals)?

*numerical answer*

In [9]:
q.answer(6, ...)

Q6: replace the placeholder with your answer, e.g. q.answer(6, 'b')


### Question 7
Hands-on: the same — self-consistent maximum Γ (meV, 2 decimals)?

*numerical answer*

In [10]:
q.answer(7, ...)

Q7: replace the placeholder with your answer, e.g. q.answer(7, 'b')


### Question 8
Hands-on: the same — mean self-consistent Γ over the grid magnons (meV, 2 decimals)?

*numerical answer*

In [11]:
q.answer(8, ...)

Q8: replace the placeholder with your answer, e.g. q.answer(8, 'b')


### Question 9
Hands-on: ⟨Γ_sc⟩/⟨Γ_Born⟩ for S = 3 (2 decimals)?

*numerical answer*

In [12]:
q.answer(9, ...)

Q9: replace the placeholder with your answer, e.g. q.answer(9, 'b')


### Question 10
Hands-on: ⟨Γ_sc⟩/⟨Γ_Born⟩ for S = 100 (2 decimals)?

*numerical answer*

In [13]:
q.answer(10, ...)

Q10: replace the placeholder with your answer, e.g. q.answer(10, 'b')


### Question 11
Hands-on: at h = 0.75 h_s (just below the decay threshold) …

* **(a)** Born and self-consistent agree
* **(b)** Born gives max Γ ≈ 1.0 meV (grid/threshold artefact), self-consistent ≈ 0.13 meV
* **(c)** both vanish exactly
* **(d)** self-consistent is larger

In [14]:
q.answer(11, ...)

Q11: replace the placeholder with your answer, e.g. q.answer(11, 'b')


### Question 12
Hands-on: triangular AFM — near-Goldstone modes with the Lorentzian line shape …

* **(a)** stay undamped
* **(b)** become overdamped (Γ/E ≈ 2) — unphysical tails of the large off-resonant weights; the Gaussian keeps Γ/E < 0.01
* **(c)** vanish
* **(d)** are unchanged

In [15]:
q.answer(12, ...)

Q12: replace the placeholder with your answer, e.g. q.answer(12, 'b')


### Question 13
Which quantity measures self-consistency?

* **(a)** the number of iterations
* **(b)** residual(): max |Γ − (−2 Im Σ(E))| with the converged internal widths
* **(c)** the energy
* **(d)** Z

In [16]:
q.answer(13, ...)

Q13: replace the placeholder with your answer, e.g. q.answer(13, 'b')


### Question 14
What is Z in the output?

* **(a)** the magnon number
* **(b)** the quasiparticle residue 1/(1 − ∂Re Σ/∂ω)
* **(c)** the coordination number
* **(d)** the partition function

In [17]:
q.answer(14, ...)

Q14: replace the placeholder with your answer, e.g. q.answer(14, 'b')


### Question 15
How are widths at off-grid momenta (band paths) obtained?

* **(a)** recomputed self-consistently
* **(b)** periodic linear interpolation of the converged grid widths for the internal lines, then one evaluation
* **(c)** set to zero
* **(d)** nearest neighbour

In [18]:
q.answer(15, ...)

Q15: replace the placeholder with your answer, e.g. q.answer(15, 'b')


### Question 16
Which CLI option switches on the self-consistent decay?

* **(a)** --relax
* **(b)** --hf T --cubic --sc-decay [--sc-grid ...]
* **(c)** --damping4
* **(d)** --spectral

In [19]:
q.answer(16, ...)

Q16: replace the placeholder with your answer, e.g. q.answer(16, 'b')


### Question 17
What does mode='dyson' do?

* **(a)** the recommended method
* **(b)** a diagnostic: real off-shell root with bare lines (mixes orders; breaks Goldstone near Goldstone modes)
* **(c)** four-magnon damping
* **(d)** HF only

In [20]:
q.answer(17, ...)

Q17: replace the placeholder with your answer, e.g. q.answer(17, 'b')


### Question 18
Which convergence check matters most for self-consistent widths?

* **(a)** none
* **(b)** the closed-grid size (and η): Γ is a BZ sum with discrete poles
* **(c)** the number of bands
* **(d)** the temperature

In [21]:
q.answer(18, ...)

Q18: replace the placeholder with your answer, e.g. q.answer(18, 'b')


### Question 19
Is the four-magnon damping Γ⁴ part of the self-consistency?

* **(a)** yes
* **(b)** no — it is added on top (on-shell); the self-consistency treats the cubic self-energy
* **(c)** it replaces Γ³
* **(d)** it is ignored

In [22]:
q.answer(19, ...)

Q19: replace the placeholder with your answer, e.g. q.answer(19, 'b')


### Question 20
What happens to the self-consistent result when S → ∞?

* **(a)** it diverges
* **(b)** it approaches the Born result (relative corrections ∝ 1/S)
* **(c)** it vanishes
* **(d)** it oscillates

In [23]:
q.answer(20, ...)

Q20: replace the placeholder with your answer, e.g. q.answer(20, 'b')


## Your score

In [24]:
q.score()

Quiz 11 — self-consistent magnon decay: 0/20 correct (0 attempted)
  not yet answered: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20


0